# WPFormer — improvements and ablation study (CrackSeg9k)

Companion to `WPFormer_CrackSeg9k_Colab.ipynb`. That notebook reproduced the paper.
**This one tests six changes against that reproduction — one of them helped.**

Final numbers are in [section 14](#14--results). Every row below was measured on one T4 at
30 epochs on a frozen split.

Everything runs through one shared file, `wpformer_plus.py`, where each improvement is a
switch that defaults to **off**. With no flags it reproduces the paper's recipe exactly,
so `--preset baseline` is a genuine baseline and every ablation row differs from the row
above it by exactly one thing.

| preset | what changes vs. the paper |
|---|---|
| `baseline` | nothing — PVTv2-B2, BCE+IoU, repo augmentation |
| `loss` | weighted BCE + weighted IoU, deep-supervision reweighting |
| `backbone` | PVTv2-B2 → **B4**, lr 8e-5 → 5e-5 |
| `aug` | + vertical flip, rot90, more rotation, colour jitter |
| `ema` | + averaged weights |
| `boundary` | `loss` + a boundary term on the repo's discarded Canny edge map |
| `full` | loss + backbone + aug + ema + TTA |

**Two things that make this tractable:** mixed precision roughly halves each run, and every
row — including the combined model — is trained for **30 epochs**. A comparison is fair when
both sides get the same budget; that budget does not have to be the paper's 60. Train the
combined model at 60 only as an extra, clearly-labelled row, because a 60-epoch model beating
30-epoch rows has won on budget rather than on method.

**Measured cost:** **6.3 min/epoch** on PVTv2-B2 and **10.1 min/epoch** on B4, so a 30-epoch
row is **3.2 h (B2)** or **5.1 h (B4)**.

**On a paid tier, stay on the T4.** A faster GPU costs more compute units per unit of work,
and — more importantly — the baseline and loss rows were measured on a T4. Keeping every row
on the same hardware removes one more thing a reviewer could question. Switch only if you are
out of time, and say so in the report.

> ⚠️ **Mount Google Drive in section 2.** Colab will disconnect long before the whole table
> is done, and anything in `/content` is lost. Everything below is built for that:
>
> * a **finished** run (its `summary.json` exists) is **skipped**, so re-running the notebook
>   never repeats completed work;
> * an **interrupted** run **resumes** from its last saved state — model, optimiser,
>   scheduler, scaler and EMA — instead of starting over.
>
> After a disconnect: reconnect, run sections 1–8, then re-run whichever experiment cell was
> in flight. It picks up where it stopped.

## 1 · GPU

In [ ]:
import torch, sys, platform
print("python:", sys.version.split()[0], "| torch:", torch.__version__)
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> T4 GPU"
print("gpu   :", torch.cuda.get_device_name(0),
      f"({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")

## 2 · Mount Drive

Runs and checkpoints are written here so a disconnect costs one epoch, not the whole run.

In [ ]:
import os
try:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/wpformer_runs"
except Exception as e:
    print("Drive not mounted, falling back to local (will be lost on disconnect):", e)
    OUT_DIR = "/content/runs"
os.makedirs(OUT_DIR, exist_ok=True)
print("runs ->", OUT_DIR)

## 3 · Clone the repo, install, apply the compatibility shims

In [ ]:
import os, sys, subprocess, types, textwrap

REPO_DIR, COMMIT = "/content/WPFormer", "83a33bbf5ed96dff069e9d58f5f3e0c464bae446"
if not os.path.isdir(REPO_DIR):
    subprocess.run(["git","clone","-q","https://github.com/fengyan-cv/WPFormer.git",REPO_DIR], check=True)
subprocess.run(["git","-C",REPO_DIR,"checkout","-q",COMMIT], check=True)
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)

!pip install -q gdown timm

import torch, timm, timm.models
# timm moved these module paths
try:
    from timm.models.layers import DropPath
except Exception:
    import timm.layers as _tl
    m = types.ModuleType("timm.models.layers")
    for n in dir(_tl):
        if not n.startswith("__"): setattr(m, n, getattr(_tl, n))
    sys.modules["timm.models.layers"] = m; timm.models.layers = m
try:
    from timm.models.registry import register_model
except Exception:
    try: from timm.models._registry import register_model as _rm
    except Exception: from timm.models import register_model as _rm
    m = types.ModuleType("timm.models.registry"); m.register_model = _rm
    sys.modules["timm.models.registry"] = m; timm.models.registry = m

# mmcv is imported at the top of defect_test.py and never called
SHIM = "/content/_shims"
try:
    from mmcv.cnn import get_model_complexity_info
except Exception:
    os.makedirs(f"{SHIM}/mmcv/cnn", exist_ok=True)
    open(f"{SHIM}/mmcv/__init__.py","w").write("")
    open(f"{SHIM}/mmcv/cnn/__init__.py","w").write(
        "def get_model_complexity_info(model, input_shape, *a, **kw):\n"
        "    n = sum(p.numel() for p in model.parameters())\n"
        "    return 'n/a (stub)', f'{n/1e6:.2f} M'\n")
    sys.path.insert(0, SHIM)

# PyTorch >= 2.6 flipped this default
if not getattr(torch.load, "_patched", False):
    _o = torch.load
    def _l(*a, **k):
        k.setdefault("weights_only", False); return _o(*a, **k)
    _l._patched = True; torch.load = _l

print("[ok] repo + shims ready at", os.getcwd())

## 4 · Fetch the training script

`wpformer_plus.py` holds every improvement behind a switch. It lives in the project
repository, so it is downloaded rather than pasted here — one source of truth, and this
notebook stays readable.

Source: [wpformer_plus.py](https://github.com/Yuvraj0208/WPFormer-Surface-Defect-Detection/blob/main/wpformer_plus.py)

In [ ]:
import os, urllib.request

URL = "https://raw.githubusercontent.com/Yuvraj0208/WPFormer-Surface-Defect-Detection/main/wpformer_plus.py"
DST = "/content/WPFormer/wpformer_plus.py"
urllib.request.urlretrieve(URL, DST)
print(f"{os.path.getsize(DST):,} bytes -> {DST}")

## 5 · Download weights and data

Four things. `pvt_v2_b4.pth` is the extra one — it's what the backbone upgrade needs.
All IDs come from the official README table.

In [ ]:
import os, gdown, glob, zipfile, tarfile

CKPT, DL = "/content/checkpoints", "/content/downloads"
os.makedirs(CKPT, exist_ok=True); os.makedirs(DL, exist_ok=True)

IDS = {
    "pvt_v2_b2.pth":  "1o3PDfaIKlx1EB21lbt_h37nRzwzJYoIX",   # backbone, paper
    "pvt_v2_b4.pth":  "1z_hZm-6M8lUxpBCbEiLa0TqY8FzjOX3z",   # backbone, upgrade
    "CrackSeg9k.pth": "17Yq3nr3CoxGL0P6hXdWnWmDo3yiCYzVU",   # authors' trained model
}
for fn, fid in IDS.items():
    p = os.path.join(CKPT, fn)
    if os.path.exists(p) and os.path.getsize(p) > 1e6:
        print("cached:", fn); continue
    if not gdown.download(id=fid, output=p, quiet=False, resume=True):
        raise RuntimeError(f"Drive download failed for {fn} (id={fid}) - "
                           f"likely the daily quota. Copy it to your own Drive and retry.")

ARCH = (".zip",".tar",".gz",".tgz")
have = [p for p in glob.glob(f"{DL}/*")
        if os.path.isfile(p) and p.lower().endswith(ARCH) and os.path.getsize(p) > 1e6]
archive = have[0] if have else gdown.download(
    id="1pOQBOjs_r9g6by0QQWU6hFT-dGeHlQqZ", output=DL+"/", quiet=False, resume=True)

RAW = "/content/datasets_raw"
if not os.path.isdir(RAW) or not os.listdir(RAW):
    os.makedirs(RAW, exist_ok=True)
    (zipfile.ZipFile(archive) if zipfile.is_zipfile(archive)
     else tarfile.open(archive)).extractall(RAW)
print("\nextracted ->", RAW)

## 6 · Normalise **both** splits

The first notebook only needed the test split. Training needs `train` too, so this finds
both and exposes them as `/content/data/CrackSeg9k/{train,test}/{images,gt}`.

Expect **7243 train / 395 test**. A different count means a different dataset version, and
your numbers stop being comparable to the paper.

In [ ]:
import os
EXTS = (".jpg",".jpeg",".png",".bmp",".tif",".tiff",".JPG",".PNG")
IMG_N = ["images","image","img","imgs","Images","Image","JPEGImages"]
GT_N  = ["gt","GT","gts","masks","mask","labels","label","annotations","Masks"]

def n_img(d): return len([f for f in os.listdir(d) if f.endswith(EXTS)]) if os.path.isdir(d) else 0

def find_split(root, key):
    best = []
    for dp, dn, _ in os.walk(root):
        if key.lower() not in os.path.basename(dp).lower(): continue
        i = next((os.path.join(dp,n) for n in IMG_N if n_img(os.path.join(dp,n))>0), None)
        g = next((os.path.join(dp,n) for n in GT_N  if n_img(os.path.join(dp,n))>0), None)
        if i and g: best.append((n_img(i), i, g))
    best.sort(reverse=True)
    return (best[0][1], best[0][2]) if best else None

DATA = "/content/data/CrackSeg9k"
for split, expect in (("train", 7243), ("test", 395)):
    found = find_split(RAW, split)
    if found is None:
        raise RuntimeError(f"could not find the '{split}' split under {RAW}")
    src_i, src_g = found
    for link, tgt in (("images", src_i), ("gt", src_g)):
        dst = os.path.join(DATA, split, link)
        os.makedirs(os.path.dirname(dst), exist_ok=True)
        if os.path.islink(dst): os.unlink(dst)
        if not os.path.exists(dst): os.symlink(os.path.abspath(tgt), dst)
    n = n_img(os.path.join(DATA, split, "images"))
    flag = "ok" if n == expect else f"WARNING expected {expect}"
    print(f"{split:6s} {n:5d} images   [{flag}]")

## 7 · Self-test — run this before anything long

~20 seconds. Checks the new losses actually behave like losses (a good prediction must
score lower than a bad one), that gradients flow, that nothing produces NaN on an empty
mask, that EMA does the arithmetic it claims, and — the easiest thing to get silently
wrong — that **TTA undoes its flips correctly**.

If any line says FAIL, fix it before spending two hours training.

In [ ]:
!python wpformer_plus.py --selftest

## 8 · Freeze the validation split

CrackSeg9k ships one train/test split and **no validation set**. Selecting epochs or
thresholds on the test set would invalidate every comparison in the report, so we carve
10% out of *train*, once, with a fixed seed, and write the exact filenames to a JSON file.

**Commit that JSON to your repo.** It is what makes your three members' results comparable.

In [ ]:
import sys, json, os
sys.path.insert(0, "/content/WPFormer")
os.chdir("/content/WPFormer")

from wpformer_plus import Config, freeze_val_split
cfg = Config(data_root="/content/data/CrackSeg9k",
             split_file=os.path.join(OUT_DIR, "val_split.json"))
tr, va = freeze_val_split(cfg)
print(f"\ntrain {len(tr)}  |  val {len(va)}")
print("split file:", cfg.split_file)

## 9 · The experiments

Each cell is one row of the ablation table. **Run them one at a time** — about 3–3.5 hours
each at 30 epochs. Results land in `OUT_DIR/<name>/summary.json` on Drive.

Every run prints per-epoch validation wF so you can watch it converge, then evaluates the
best checkpoint on the test set.

**Safe to re-run.** A completed row prints `ALREADY FINISHED, skipping` and returns
immediately; an interrupted row prints `RESUMED from epoch N`. Neither wastes GPU time. To
deliberately redo a finished row, add `--force`.

If you are short on wall-clock time, `--val-every 2` cuts roughly 20% — but use the same
value for **every** row, or the rows stop being comparable.

In [ ]:
# shared arguments, kept in one variable so every row below is identical except the preset
ARGS = f'--data-root /content/data/CrackSeg9k --out-dir "{OUT_DIR}"'
print(ARGS, "\n")

# where each row currently stands
import os, json
for name in ["baseline", "loss", "backbone", "aug", "ema", "boundary", "full"]:
    d = os.path.join(OUT_DIR, name)
    summ, state, hist = (os.path.join(d, f) for f in
                         ("summary.json", "state.pt", "history.json"))
    if os.path.exists(summ):
        wf = json.load(open(summ))["test"]["wFmeasure"]
        print(f"  {name:10s} DONE      test wF {wf:.4f}")
    elif os.path.exists(state) or os.path.exists(hist):
        ep = json.load(open(hist))[-1]["epoch"] if os.path.exists(hist) else "?"
        print(f"  {name:10s} PARTIAL   through epoch {ep} -- re-run to resume")
    else:
        print(f"  {name:10s} not started")

### 9.1 · Run every row

With **background execution** on (Colab Pro: Runtime menu; it keeps running after you close
the tab) the cell below works through every remaining row in one go. Finished rows are
skipped, interrupted rows resume, and a row that fails does not stop the queue.

Start it, close the laptop, come back to a filled-in table.

**Do not change `--batch-size` or `--lr`.** The baseline and loss rows are already measured
at batch 4; changing either makes every later row incomparable to them.

In [ ]:
import time
QUEUE = [("backbone", 30), ("aug", 30), ("ema", 30), ("loss_only", 30), ("full", 30)]

for name, ep in QUEUE:
    print(f"\n{'#'*72}\n#  {name}   ({ep} epochs)\n{'#'*72}", flush=True)
    t0 = time.time()
    !python wpformer_plus.py --preset {name} --epochs {ep} {ARGS}
    print(f"#  {name} finished in {(time.time()-t0)/3600:.2f} h", flush=True)

print("\n\nqueue complete -- run section 11 to build the table")

## 10 · Test-time augmentation and the min–max question

No training — every configuration below is evaluated on the same finished checkpoint, so
these rows cost minutes rather than hours.

TTA averages several transformed views of each image. The number of views that the model
was **never trained on** turns out to decide whether it helps: the repo's augmentation
flips horizontally only and never rescales, so vertical flips and rescaled inputs are out
of distribution for every model here.

The min–max question is separate. `defect_test.py` stretches each prediction so its darkest
pixel becomes 0 and its brightest 1; on an image where the model is *correctly* unsure that
turns quiet uncertainty into a confident false positive. It costs two minutes to test.

In [ ]:
import os
CKPT = os.path.join(OUT_DIR, "backbone", "best.pth")     # the best model we trained
EVAL = f'--preset backbone --eval-only --ckpt "{CKPT}" --data-root /content/data/CrackSeg9k'

print("=== no TTA ===")
!python wpformer_plus.py {EVAL}

print("\n=== TTA: identity + horizontal flip  (0 of 2 views unseen) ===")
!python wpformer_plus.py {EVAL} --tta --tta-views 2 --tta-scales 1.0

print("\n=== TTA: + vertical flip  (2 of 4 views unseen) ===")
!python wpformer_plus.py {EVAL} --tta --tta-views 4 --tta-scales 1.0

print("\n=== TTA: + scales 0.75/1.25  (10 of 12 views unseen) ===")
!python wpformer_plus.py {EVAL} --tta --tta-views 4 --tta-scales 0.75,1.0,1.25

print("\n=== min-max stretch removed ===")
!python wpformer_plus.py {EVAL} --tta --tta-views 4 --no-minmax

## 11 · Build the ablation table

In [ ]:
import os, json, glob
import pandas as pd

PAPER = {"MAE": .0135, "wFmeasure": .7672, "Smeasure": .8493,
         "meanFm": .7679, "meanEm": .9481}

rows = []
for f in sorted(glob.glob(os.path.join(OUT_DIR, "*", "summary.json"))):
    s = json.load(open(f))
    t = s["test"]
    rows.append({
        "run": s["name"],
        "backbone": s["config"]["backbone"].replace("pvt_v2_", ""),
        "loss": s["config"]["loss"],
        "aug": s["config"]["aug"],
        "ema": s["config"]["ema"],
        "tta": s["config"]["tta"],
        "params(M)": s["params_M"],
        "MAE": round(t["MAE"], 4),
        "wF": round(t["wFmeasure"], 4),
        "Sm": round(t["Smeasure"], 4),
        "mF": round(t["meanFm"], 4),
        "mE": round(t["meanEm"], 4),
        "IoU": round(s["test_best_IoU"], 4),
        "Dice": round(s["test_Dice_at_best"], 4),
    })

if not rows:
    print("no finished runs yet")
else:
    df = pd.DataFrame(rows)
    base = df[df.run == "baseline"]
    if len(base):
        b = base.iloc[0]
        df["ΔwF vs baseline"] = (df["wF"] - b["wF"]).round(4)
        df["ΔwF %"] = ((df["wF"] - b["wF"]) / b["wF"] * 100).round(2)
    display(df)
    df.to_csv(os.path.join(OUT_DIR, "ablation_table.csv"), index=False)
    print("\npaper (60 ep):", {k: f"{v:.4f}" for k, v in PAPER.items()})
    print("saved ->", os.path.join(OUT_DIR, "ablation_table.csv"))

## 12 · Training curves

One line per run. If a curve is still climbing at the last epoch, that run was
budget-limited rather than converged — worth saying in the report.

In [ ]:
import matplotlib.pyplot as plt, json, glob, os

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for f in sorted(glob.glob(os.path.join(OUT_DIR, "*", "history.json"))):
    h = json.load(open(f))
    name = os.path.basename(os.path.dirname(f))
    ep = [r["epoch"] for r in h]
    ax[0].plot(ep, [r["loss"] for r in h], label=name)
    ax[1].plot(ep, [r["wFmeasure"] for r in h], label=name)
ax[0].set_title("training loss"); ax[0].set_xlabel("epoch")
ax[1].set_title("validation wF-measure"); ax[1].set_xlabel("epoch")
for a in ax: a.grid(alpha=.3); a.legend(fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "curves.png"), dpi=130, bbox_inches="tight")
plt.show()

## 13 · Baseline vs the B4 backbone, side by side

The figure the report needs: where the one change that worked actually differs.

In [ ]:
import os, glob, numpy as np, cv2, matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

A = os.path.join(OUT_DIR, "baseline", "preds")
B = os.path.join(OUT_DIR, "backbone", "preds")
GT = "/content/data/CrackSeg9k/test/gt/"
IM = "/content/data/CrackSeg9k/test/images/"

if not (os.path.isdir(A) and os.path.isdir(B)):
    print("need both the 'baseline' and 'backbone' runs finished")
else:
    gts = sorted(os.listdir(GT))
    # show the images where the two models disagree most
    diffs = []
    for g in gts[::5]:
        s = Path(g).stem
        pa, pb = os.path.join(A, s+".png"), os.path.join(B, s+".png")
        if not (os.path.exists(pa) and os.path.exists(pb)): continue
        a = cv2.imread(pa, 0).astype(float); b = cv2.imread(pb, 0).astype(float)
        diffs.append((np.abs(a-b).mean(), s))
    diffs.sort(reverse=True)
    picks = [s for _, s in diffs[:5]]

    fig, axes = plt.subplots(len(picks), 4, figsize=(13, 3.1*len(picks)))
    for r, s in enumerate(picks):
        img = np.array(Image.open(glob.glob(IM+s+".*")[0]).convert("RGB"))
        gt  = cv2.imread(glob.glob(GT+s+".*")[0], 0)
        pa  = cv2.imread(os.path.join(A, s+".png"), 0)
        pb  = cv2.imread(os.path.join(B, s+".png"), 0)
        for c, (im, t, cm) in enumerate([(img,"image",None), (gt,"ground truth","gray"),
                                         (pa,"baseline  wF .7390","gray"), (pb,"PVTv2-B4  wF .7511","gray")]):
            axes[r,c].imshow(im, cmap=cm); axes[r,c].axis("off")
            if r == 0: axes[r,c].set_title(t, fontsize=11)
    plt.suptitle("Where the B4 backbone changes the prediction", y=.995)
    plt.tight_layout()
    plt.savefig(os.path.join(OUT_DIR, "qualitative_compare.png"), dpi=130, bbox_inches="tight")
    plt.show()

## 14 · Results

Measured on the frozen split, 30 epochs per row, batch 4, one T4.

| Run | Change | wF<sub>β</sub> | ΔwF | Δ% |
|:---|:---|:---:|:---:|:---:|
| baseline | paper recipe | .7390 | — | — |
| **backbone** | **PVTv2-B2 → B4** | **.7511** | **+.0121** | **+1.64%** |
| ema | averaged weights | .7364 | −.0026 | −0.35% |
| loss_only | structure loss | .7317 | −.0073 | −0.99% |
| loss | structure loss + DS reweighting | .7286 | −.0104 | −1.41% |
| aug | vflip, rot90, colour jitter | .7111 | −.0279 | −3.78% |

**One of six changes helped.** Test-time augmentation was neutral at best, and its cost rose
with the number of averaged views the model had never seen in training (.7511 → .7498 →
.7189 → .7024). The per-image min–max stretch is neutral: .70238 with, .70243 without.

The +2–3% target was **not met**; +1.64% is reported as measured.

Every change that increased the difficulty or diversity the model had to absorb cost
accuracy, and the only one that helped added capacity instead. Best validation wF arrived at
epoch 27–30 in all six runs, so nothing converged inside the budget — a regulariser is
charged its cost immediately and repays it near convergence. That is a hypothesis consistent
with the evidence, not a proven claim; testing it needs a 60-epoch pair.

### Reporting rules held to

- compared against **our own** 30-epoch baseline, never the paper's printed 60-epoch numbers
- parameters reported next to every accuracy figure — B4 is 63.3 M against B2's 26.1 M
- IoU threshold chosen on validation, then applied fixed to test
- every row is 30 epochs; no row is quietly given a longer budget